In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

DATA_PATH = "bank_transactions_data_2_augmented_clean_2.csv"
TARGET_FRAUD_RATE = 0.04

In [11]:
df = pd.read_csv(DATA_PATH)

# Parse both date-only ("M/D/YYYY") and datetime ("M/D/YYYY H:MM") rows.
# pandas defaults missing time to 00:00:00, so Hour is always an integer (0–23),
# never NaN — rows that had no time component get Hour=0 (midnight).
df["TransactionDate"] = pd.to_datetime(
    df["TransactionDate"], format="mixed", dayfirst=False
)

df["Hour"] = df["TransactionDate"].dt.hour
df["DayOfWeek"] = df["TransactionDate"].dt.dayofweek
df["Month"] = df["TransactionDate"].dt.month
df["Year"] = df["TransactionDate"].dt.year

df = df.rename(columns={"IP Address": "IPAddress"})

df = df.sort_values(["AccountID", "TransactionDate"]).reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(f"\nHour NaN count: {df['Hour'].isna().sum()} (expected 0)")
print(f"Hour range: {df['Hour'].min()}–{df['Hour'].max()}")

(50000, 19)
TransactionID                     str
AccountID                         str
TransactionAmount             float64
TransactionDate        datetime64[us]
TransactionType                   str
Location                          str
DeviceID                          str
IPAddress                         str
MerchantID                        str
Channel                           str
CustomerAge                     int64
CustomerOccupation                str
TransactionDuration             int64
LoginAttempts                   int64
AccountBalance                float64
Hour                            int32
DayOfWeek                       int32
Month                           int32
Year                            int32
dtype: object

Hour NaN count: 0 (expected 0)
Hour range: 0–18


In [13]:
df

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IPAddress,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,Hour,DayOfWeek,Month,Year
0,TX032406,AC00001,47.68,2020-01-29,Debit,Denver,D000649,59.12.96.11,M034,Branch,25,Student,37,1,1649.92,0,2,1,2020
1,TX016121,AC00001,46.05,2020-02-16,Debit,Denver,D000649,59.12.96.11,M034,Branch,25,Student,37,1,1649.92,0,6,2,2020
2,TX031292,AC00001,44.39,2020-04-23,Debit,Denver,D000649,59.12.96.11,M034,Branch,25,Student,37,1,1649.92,0,3,4,2020
3,TX022030,AC00001,48.96,2020-06-16,Debit,Denver,D000649,59.12.96.11,M034,Branch,25,Student,37,1,1649.92,0,1,6,2020
4,TX024312,AC00001,50.06,2020-08-16,Debit,Denver,D000649,59.12.96.11,M034,Branch,25,Student,37,1,1649.92,0,6,8,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,TX024389,AC00500,143.30,2025-08-14,Debit,Charlotte,D000168,11.167.243.171,M099,ATM,51,Doctor,102,1,14453.35,0,3,8,2025
49996,TX022855,AC00500,132.81,2025-10-01,Debit,San Jose,D000219,100.137.90.188,M026,Online,59,Doctor,165,1,14852.42,0,2,10,2025
49997,TX009452,AC00500,152.88,2025-10-20,Debit,Charlotte,D000168,11.167.243.171,M099,ATM,51,Doctor,102,1,14453.35,0,0,10,2025
49998,TX003594,AC00500,157.98,2025-12-08,Debit,Charlotte,D000168,11.167.243.171,M099,ATM,51,Doctor,102,1,14453.35,0,0,12,2025


In [3]:
total_rows = len(df)
print(f"Total rows: {total_rows:,}\n")
print(f"{'Column':<25} {'Non-Null':>10} {'Coverage':>10}")
print("-" * 47)
for col in df.columns:
    non_null = df[col].notna().sum()
    pct = non_null / total_rows * 100
    flag = "" if pct == 100 else "  ⚠️"
    print(f"{col:<25} {non_null:>10,} {pct:>9.2f}%{flag}")

print(f"\n{'Column':<25} {'Unique':>10} {'Dtype':<15}")
print("-" * 52)
for col in df.columns:
    print(f"{col:<25} {df[col].nunique():>10,} {str(df[col].dtype):<15}")

Total rows: 50,000

Column                      Non-Null   Coverage
-----------------------------------------------
TransactionID                 50,000    100.00%
AccountID                     50,000    100.00%
TransactionAmount             50,000    100.00%
TransactionDate               50,000    100.00%
TransactionType               50,000    100.00%
Location                      50,000    100.00%
DeviceID                      50,000    100.00%
IPAddress                     50,000    100.00%
MerchantID                    50,000    100.00%
Channel                       50,000    100.00%
CustomerAge                   50,000    100.00%
CustomerOccupation            50,000    100.00%
TransactionDuration           50,000    100.00%
LoginAttempts                 50,000    100.00%
AccountBalance                50,000    100.00%
Hour                          50,000    100.00%
DayOfWeek                     50,000    100.00%
Month                         50,000    100.00%
Year                

In [ ]:
assert df["Hour"].isna().sum() == 0, "Hour has unexpected NaN values"
print("Hour coverage: 100% — no imputation needed.")
print("pandas defaulted date-only rows to 00:00:00, giving Hour=0 (midnight) for those rows.")
print(f"\nHour value distribution:")
print(df["Hour"].value_counts().sort_index().to_string())

In [ ]:
acct = df.groupby("AccountID").agg(
    acct_mean_amount=("TransactionAmount", "mean"),
    acct_std_amount=("TransactionAmount", "std"),
    acct_median_amount=("TransactionAmount", "median"),
    acct_txn_count=("TransactionID", "count"),
    acct_unique_locations=("Location", "nunique"),
    acct_unique_devices=("DeviceID", "nunique"),
    acct_unique_ips=("IPAddress", "nunique"),
    acct_unique_merchants=("MerchantID", "nunique"),
).reset_index()

acct["acct_std_amount"] = acct["acct_std_amount"].fillna(0).replace(0, 1)

df = df.merge(acct, on="AccountID", how="left")
print(f"Account aggregates merged. New shape: {df.shape}")

In [ ]:
ip_sharing = df.groupby("IPAddress")["AccountID"].nunique().reset_index()
ip_sharing.columns = ["IPAddress", "ip_acct_count"]

dev_sharing = df.groupby("DeviceID")["AccountID"].nunique().reset_index()
dev_sharing.columns = ["DeviceID", "dev_acct_count"]

df = df.merge(ip_sharing, on="IPAddress", how="left")
df = df.merge(dev_sharing, on="DeviceID", how="left")
print(f"IP/Device sharing merged. New shape: {df.shape}")

In [ ]:
df["amount_zscore"] = (
    (df["TransactionAmount"] - df["acct_mean_amount"]) / df["acct_std_amount"]
)

df["amount_to_balance_ratio"] = df["TransactionAmount"] / df["AccountBalance"].replace(0, 1)

print(f"amount_zscore range: [{df['amount_zscore'].min():.2f}, {df['amount_zscore'].max():.2f}]")
print(f"amount_to_balance_ratio range: [{df['amount_to_balance_ratio'].min():.4f}, {df['amount_to_balance_ratio'].max():.4f}]")

In [ ]:
df["prev_txn_date"] = df.groupby("AccountID")["TransactionDate"].shift(1)
df["hours_since_prev"] = (
    (df["TransactionDate"] - df["prev_txn_date"]).dt.total_seconds() / 3600
)

df["daily_txn_count"] = df.groupby(
    ["AccountID", df["TransactionDate"].dt.date]
)["TransactionID"].transform("count")

print(f"hours_since_prev: {df['hours_since_prev'].describe()}")
print(f"\ndaily_txn_count value counts:\n{df['daily_txn_count'].value_counts().sort_index().head(10)}")

In [ ]:
acct_location_freq = (
    df.groupby(["AccountID", "Location"])["TransactionID"]
    .count()
    .reset_index()
    .rename(columns={"TransactionID": "acct_loc_freq"})
)
df = df.merge(acct_location_freq, on=["AccountID", "Location"], how="left")

df["acct_loc_pct"] = df["acct_loc_freq"] / df["acct_txn_count"]

print(f"Location consistency merged. New shape: {df.shape}")
print(f"acct_loc_pct range: [{df['acct_loc_pct'].min():.4f}, {df['acct_loc_pct'].max():.4f}]")

In [ ]:
df["R01_high_login"] = (df["LoginAttempts"] >= 3).astype(int)
df["R02_very_high_login"] = (df["LoginAttempts"] >= 5).astype(int)
df["R03_amount_outlier"] = (df["amount_zscore"] > 2.5).astype(int)
df["R04_draining_balance"] = (df["amount_to_balance_ratio"] > 0.8).astype(int)
df["R05_shared_ip"] = (df["ip_acct_count"] >= 3).astype(int)
df["R06_shared_device"] = (df["dev_acct_count"] >= 3).astype(int)
df["R07_rapid_fire"] = (df["hours_since_prev"] < 0.5).astype(int)
df["R08_high_daily_freq"] = (df["daily_txn_count"] >= 5).astype(int)
df["R09_unusual_location"] = (
    (df["acct_loc_pct"] < 0.05) & (df["acct_txn_count"] >= 10)
).astype(int)
df["R10_nighttime"] = (
    df["Hour"].notna() & ((df["Hour"] < 5) | (df["Hour"] >= 23))
).astype(int)
df["R11_high_online"] = (
    (df["Channel"] == "Online") & (df["TransactionAmount"] > 800)
).astype(int)

rule_cols = [c for c in df.columns if c.startswith("R") and c[1:3].isdigit()]
print("Rule trigger counts:")
for col in rule_cols:
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"  {col}: {count:,} ({pct:.2f}%)")

In [ ]:
WEIGHTS = {
    "R01_high_login": 25,
    "R02_very_high_login": 15,
    "R03_amount_outlier": 20,
    "R04_draining_balance": 15,
    "R05_shared_ip": 15,
    "R06_shared_device": 15,
    "R07_rapid_fire": 15,
    "R08_high_daily_freq": 10,
    "R09_unusual_location": 10,
    "R10_nighttime": 5,
    "R11_high_online": 10,
}

weight_values = np.array([WEIGHTS[c] for c in rule_cols])

df["fraud_score_raw"] = df[rule_cols].values @ weight_values

max_possible = weight_values.sum()
df["fraud_score"] = (df["fraud_score_raw"] / max_possible * 100).clip(0, 100)

outlier_bonus = (
    (df["amount_zscore"].clip(0, 5) / 5 * 5) +
    (df["amount_to_balance_ratio"].clip(0, 2) / 2 * 5)
)
df["fraud_score"] = (df["fraud_score"] + outlier_bonus).clip(0, 100)

threshold = df["fraud_score"].quantile(1 - TARGET_FRAUD_RATE)
print(f"Threshold at {TARGET_FRAUD_RATE:.0%} fraud rate: {threshold:.2f}")

df["IsFraud"] = (df["fraud_score"] >= threshold).astype(int)

actual_rate = df["IsFraud"].mean()
print(f"Actual fraud rate: {actual_rate:.2%} ({df['IsFraud'].sum():,} transactions)")

In [ ]:
np.random.seed(42)

fraud_indices = df[df["IsFraud"] == 1].index
fn_rate = 0.05
fn_mask = np.random.random(len(fraud_indices)) < fn_rate
df.loc[fraud_indices[fn_mask], "IsFraud"] = 0
print(f"Flipped {fn_mask.sum()} fraud labels to 0 (false negatives)")

legit_indices = df[
    (df["IsFraud"] == 0) & (df["fraud_score"] > threshold * 0.5)
].index
fp_rate = 0.005
fp_mask = np.random.random(len(legit_indices)) < fp_rate
df.loc[legit_indices[fp_mask], "IsFraud"] = 1
print(f"Flipped {fp_mask.sum()} legit labels to 1 (false positives)")

In [ ]:
print(f"Final fraud rate: {df['IsFraud'].mean():.2%}")
assert 0.02 <= df["IsFraud"].mean() <= 0.06, "Fraud rate out of expected range"

print("\nFraud score distribution:")
print(df.groupby("IsFraud")["fraud_score"].describe())

print("\nRule trigger rates (fraud vs legit):")
for col in rule_cols:
    fraud_rate_r = df.loc[df["IsFraud"] == 1, col].mean()
    legit_rate_r = df.loc[df["IsFraud"] == 0, col].mean()
    lift = fraud_rate_r / legit_rate_r if legit_rate_r > 0 else float("inf")
    print(f"  {col}: fraud={fraud_rate_r:.2%}, legit={legit_rate_r:.2%}, lift={lift:.1f}x")

print("\nFraud by channel:")
print(pd.crosstab(df["Channel"], df["IsFraud"], normalize="index"))

print("\nFraud by transaction type:")
print(pd.crosstab(df["TransactionType"], df["IsFraud"], normalize="index"))

print("\nRule coverage of fraud cases:")
for col in rule_cols:
    coverage = df.loc[df["IsFraud"] == 1, col].mean()
    print(f"  {col} covers {coverage:.0%} of fraud")
    assert coverage < 0.95, f"Rule {col} covers too much fraud — labels are too deterministic"

print("\nAll validations passed.")

In [ ]:
original_cols = [
    "TransactionID", "AccountID", "TransactionAmount", "TransactionDate",
    "TransactionType", "Location", "DeviceID", "IPAddress", "MerchantID",
    "Channel", "CustomerAge", "CustomerOccupation", "TransactionDuration",
    "LoginAttempts", "AccountBalance",
]

output = df[original_cols + ["IsFraud", "fraud_score"]].copy()
output = output.rename(columns={"IPAddress": "IP Address"})

output = output.sort_values("TransactionID").reset_index(drop=True)

OUTPUT_PATH = "bank_transactions_data_2_augmented_clean_2_labeled.csv"
output.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {len(output):,} rows to {OUTPUT_PATH}")

check = pd.read_csv(OUTPUT_PATH)
print(f"\nSpot-check reload:")
print(f"  Shape: {check.shape}")
print(f"  Columns: {list(check.columns)}")
print(f"  Fraud rate: {check['IsFraud'].mean():.2%} ({check['IsFraud'].sum():,} fraud transactions)")